# **Módulo 2: Análisis Exploratorio de Datos (EDA)**

Analicemos el siguiente conjunto de datos:

**Contexto empresarial**. Los préstamos entre pares (P2P) en línea han facilitado la práctica de pedir y prestar dinero. En esta forma de préstamo, no hay una entrevista en persona y el prestatario puede simplemente completar un formulario en línea y obtener la aprobación del préstamo. La información proporcionada únicamente por el prestatario es propensa a exageraciones y distorsiones, especialmente cuando se trata de ingresos. Todas las empresas de préstamos entre pares se basan en un procedimiento bien diseñado que rechaza a los prestatarios con una alta probabilidad de no pagar sus deudas.

Rechazar a cualquier persona que no tenga una fuente de ingresos verificada es una política relevante que las plataformas de préstamos pueden implementar para ayudar a reducir la tasa de préstamos incobrables. Es natural sospechar que si no se puede verificar la fuente de ingresos de una persona, es posible que no pueda pagar el préstamo. Sin embargo, desde la perspectiva de un prestatario, el proceso de verificación puede ser engorroso y llevar mucho tiempo, y es posible que simplemente cambie de plataforma debido a este inconveniente.

**Problema empresarial**. Como científico de datos de una empresa emergente de préstamos P2P, debe responder a la siguiente pregunta: "¿Debería la empresa verificar la fuente de ingresos de un solicitante de préstamo en línea antes de aprobar su préstamo?"

**Contexto analítico**. Los datos son de [LendingClub (LC) Statistics](https://www.lendingclub.com/info/download-data.action) y contienen todos los préstamos emitidos entre 2007 y 2012 junto con su estado actual (pagados en su totalidad o cancelados). Hay aproximadamente 50 variables que describen a los prestatarios y los préstamos; con el fin de reducir la complejidad, la empresa ya ha realizado una preselección de estas variables en función de los análisis existentes de LendingClub para seleccionar nueve variables relevantes, como los ingresos anuales, la calificación crediticia de LendingClub, la propiedad de la vivienda, etc. Utilizaremos una nueva técnica, la regresión logística , para responder a nuestra pregunta en cuestión.

Echemos un vistazo a los datos que tenemos a nuestra disposición:

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# Cargar paquetes
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv("_data/Lending_club_cleaned_2.csv")
df.head()

,loan_status,annual_inc,verification_status,emp_length,home_ownership,int_rate,loan_amnt,purpose,term,grade
0,Fully Paid,24000.0,Verified,10+ years,RENT,10.65%,5000,credit_card,36 months,B
1,Charged Off,30000.0,Source Verified,< 1 year,RENT,15.27%,2500,car,60 months,C
2,Fully Paid,12252.0,Not Verified,10+ years,RENT,15.96%,2400,small_business,36 months,C
3,Fully Paid,49200.0,Source Verified,10+ years,RENT,13.49%,10000,other,36 months,C
4,Fully Paid,80000.0,Source Verified,1 year,RENT,12.69%,3000,other,60 months,B


In [5]:
# informacion general del dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38705 entries, 0 to 38704
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   loan_status          38705 non-null  object 
 1   annual_inc           38705 non-null  float64
 2   verification_status  38705 non-null  object 
 3   emp_length           38705 non-null  object 
 4   home_ownership       38705 non-null  object 
 5   int_rate             38705 non-null  object 
 6   loan_amnt            38705 non-null  int64  
 7   purpose              38705 non-null  object 
 8   term                 38705 non-null  object 
 9   grade                38705 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 3.0+ MB


In [6]:
#Cambiar loan_status, verification_status, emp_length, term y grade al tipo categórico
df.loan_status = df.loan_status.astype(pd.api.types.CategoricalDtype(categories=['Charged Off', 'Fully Paid']))
df.verification_status = df.verification_status.astype(pd.api.types.CategoricalDtype(categories=['Not Verified', 'Source Verified', 'Verified']))
df.emp_length = df.emp_length.astype(pd.api.types.CategoricalDtype(categories=['< 1 year', '1 year', '2 years', '3 years', '4 years', \
                                                             '5 years', '6 years', '7 years', '8 years', '9 years', \
                                                             '10+ years']))
df.home_ownership = df.home_ownership.astype(pd.api.types.CategoricalDtype(categories=['RENT','MORTGAGE','OWN','OTHER']))
df.term = df.term.astype(pd.api.types.CategoricalDtype(categories=[' 36 months', ' 60 months']))
df.grade = df.grade.astype(pd.api.types.CategoricalDtype(categories=['A','B','C','D','E','F','G']))
df.purpose = df.purpose.astype(pd.api.types.CategoricalDtype(categories=['debt_consolidation', 'credit_card', 'other', 'home_improvement',
                                                                         'major_purchase', 'small_business', 'car', 'wedding', 'medical',
                                                                         'moving', 'house', 'vacation', 'educational', 'renewable_energy']))

#Además, los datos originales en int_rate contienen cadenas en la forma 'x.xx%',
#eliminamos el % y cambiamos los valores a tipo float:
df.int_rate = df.int_rate.str.rstrip('%').astype('float')

In [10]:
# muestra las 5 primeras filas del dataframe
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38705 entries, 0 to 38704
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   loan_status          38705 non-null  category
 1   annual_inc           38705 non-null  float64 
 2   verification_status  38705 non-null  category
 3   emp_length           38705 non-null  category
 4   home_ownership       38705 non-null  category
 5   int_rate             38705 non-null  float64 
 6   loan_amnt            38705 non-null  int64   
 7   purpose              38705 non-null  category
 8   term                 38705 non-null  category
 9   grade                38705 non-null  category
dtypes: category(7), float64(2), int64(1)
memory usage: 1.1 MB


In [11]:
# conteo
df['loan_status'].value_counts()

loan_status
Fully Paid     33265
Charged Off     5440
Name: count, dtype: int64

In [12]:
# Proporciones
df['loan_status'].value_counts(normalize=True)

loan_status
Fully Paid     0.85945
Charged Off    0.14055
Name: proportion, dtype: float64

In [13]:
# Descriptivo de una sola variable categórica solo conteo
conteo = df['loan_status'].value_counts()

# Descriptivo de una sola variable categórica solo proporcion
porcentajes = df['loan_status'].value_counts(normalize=True)*100

# Combinar los resultados en un solo DataFrame
resultado = pd.DataFrame({
    'Conteo': conteo,
    'Proporción': porcentajes
})

# imprimir
print(resultado)

             Conteo  Proporción
loan_status                    
Fully Paid    33265   85.944968
Charged Off    5440   14.055032


Veamos otra variable categórica, es decir, las independientes las que me ayudan a predecir el modelo

In [14]:
# Descriptivo de una sola variable categórica solo conteo
conteo = df['verification_status'].value_counts()

# Descriptivo de una sola variable categórica solo proporcion
porcentajes = df['verification_status'].value_counts(normalize=True)*100

# Combinar los resultados en un solo DataFrame
resultado = pd.DataFrame({
    'Conteo': conteo,
    'Proporción': porcentajes
})

# imprimir
print(resultado)

                     Conteo  Proporción
verification_status                    
Not Verified          16499   42.627567
Verified              12387   32.003617
Source Verified        9819   25.368815


Unamos todas las medidas

In [15]:
import pandas as pd
import numpy as np

# en el ingreso anual



# Construir la serie con todos los estadísticos
estadisticos = pd.Series({
    'Media': df['annual_inc'].mean(),
    'Desviación Estándar': df['annual_inc'].std(),
    'Mediana': df['annual_inc'].median(),
    'Primer Cuartil (Q1)': df['annual_inc'].quantile(0.25),
    'Tercer Cuartil (Q3)': df['annual_inc'].quantile(0.75),
    'Rango Intercuartílico (IQR)': df['annual_inc'].quantile(0.75) - df['annual_inc'].quantile(0.25),
    'Mínimo': df['annual_inc'].min(),
    'Máximo': df['annual_inc'].max(),
    'Coeficiente de Asimetría': df['annual_inc'].skew(),
    'Curtosis': df['annual_inc'].kurtosis()
})

print(estadisticos)

Media                          6.961750e+04
Desviación Estándar            6.422378e+04
Mediana                        6.000000e+04
Primer Cuartil (Q1)            4.149600e+04
Tercer Cuartil (Q3)            8.320000e+04
Rango Intercuartílico (IQR)    4.170400e+04
Mínimo                         4.000000e+03
Máximo                         6.000000e+06
Coeficiente de Asimetría       3.107386e+01
Curtosis                       2.298846e+03
dtype: float64


In [18]:
# resumen estadístico de una variable numérica
df['annual_inc'].describe()

df.describe()


,annual_inc,int_rate,loan_amnt
count,3.870500e+04,38705.000000,38705.000000
mean,6.961750e+04,12.059525,11303.916161
std,6.422378e+04,3.719517,7470.319733
min,4.000000e+03,5.420000,500.000000
25%,4.149600e+04,9.320000,5500.000000
50%,6.000000e+04,11.860000,10000.000000
75%,8.320000e+04,14.590000,15000.000000
max,6.000000e+06,24.590000,35000.000000


In [21]:
# transformar la variable numérica 'annual_inc' logarítmicamente para reducir la asimetría y la curtosis
df['annual_inc_log'] = np.log(df['annual_inc'])
df['annual_inc_log'].describe()

count    38705.000000
mean        10.985480
std          0.551504
min          8.294050
25%         10.633352
50%         11.002100
75%         11.329003
max         15.607270
Name: annual_inc_log, dtype: float64

### **2.1.4. Descriptivo bivariado**

#### **Variable categórica vs variable numérica**

In [22]:
# descriptivo ingreso anual según la suscripcion solo una variable

df.groupby('loan_status')['annual_inc'].mean()

var_num_cat = df.groupby('loan_status')['annual_inc'].agg([
    ('Promedio', 'mean'),                       # Media
    ('Desv Estandar', 'std'),                         # Desviación estándar
    ('Minimo', 'min'),                         # Mínimo
    ('Maximo', 'max'),                         # Máximo
    ('Mediana', 'median'),                   # Mediana
    ('Cuantil 1', lambda x: x.quantile(0.25)),   # Cuantil 25%
    ('Cuantil 3', lambda x: x.quantile(0.75))    # Cuantil 75%
])

# Mostrar los resultados
print(var_num_cat)

                 Promedio  Desv Estandar  Minimo     Maximo  Mediana  \
loan_status                                                            
Charged Off  63438.852340   48040.293268  4080.0  1250000.0  54000.0   
Fully Paid   70627.921006   66442.631570  4000.0  6000000.0  60000.0   

             Cuantil 1  Cuantil 3  
loan_status                        
Charged Off    38400.0    75000.0  
Fully Paid     42000.0    85000.0  


In [23]:
df.groupby('loan_status')['annual_inc'].describe()

,count,mean,std,min,25%,50%,75%,max
loan_status,,,,,,,,
Charged Off,5440.0,63438.852340,48040.293268,4080.0,38400.0,54000.0,75000.0,1250000.0
Fully Paid,33265.0,70627.921006,66442.631570,4000.0,42000.0,60000.0,85000.0,6000000.0


#### **Variable categórica vs variable categórica**

In [24]:
# Crear la tabla pivote con conteo de personas
pivot_table = pd.pivot_table(df,
                             index='loan_status',
                             columns='verification_status',
                             aggfunc='size')
print(pivot_table)

# Otra forma de crear una tabla de contingencia
crosstab = pd.crosstab(df['loan_status'],
                       df['verification_status'],
                       margins=True, margins_name='Total')
print(crosstab)

verification_status  Not Verified  Source Verified  Verified
loan_status                                                 
Charged Off                  2050             1413      1977
Fully Paid                  14449             8406     10410
verification_status  Not Verified  Source Verified  Verified  Total
loan_status                                                        
Charged Off                  2050             1413      1977   5440
Fully Paid                  14449             8406     10410  33265
Total                       16499             9819     12387  38705


In [25]:
crosstab = pd.crosstab(df['loan_status'],
                       df['verification_status'])

crosstab_pct = crosstab.div(crosstab.sum(axis=0), axis=1).mul(100).round(2)

crosstab_combined = crosstab.astype(str) + ' (' + crosstab_pct.astype(str) + '%)'
print(crosstab_combined)

verification_status    Not Verified Source Verified        Verified
loan_status                                                        
Charged Off           2050 (12.42%)   1413 (14.39%)   1977 (15.96%)
Fully Paid           14449 (87.58%)   8406 (85.61%)  10410 (84.04%)


#### **Correlación**



In [ ]:
# Importemos
from scipy.stats import spearmanr
from scipy.stats import pearsonr

# Seleccionemos las variables numéricas
df_num = df.select_dtypes(include=[np.number])

# Calcular la correlación de Spearman para cada par de columnas
spearman_corr = df_num.corr(method='spearman')

pearson_corr = df_num.corr(method='pearson')

# Mostrar la matriz de correlación de Spearman
print("Matriz de correlación de Spearman:")
print(spearman_corr)

# Mostrar la matriz de correlación de Pearson
print("Matriz de correlación de Pearson:")
print(pearson_corr)

## No hay correlacion alta
## y las matrices de correlacion son simétricas, lo que significa que la correlación entre dos variables es la misma sin importar el orden en que se consideren.


Matriz de correlación de Spearman:
                annual_inc  int_rate  loan_amnt  annual_inc_log
annual_inc        1.000000   0.05583   0.425268        1.000000
int_rate          0.055830   1.00000   0.253590        0.055830
loan_amnt         0.425268   0.25359   1.000000        0.425268
annual_inc_log    1.000000   0.05583   0.425268        1.000000
Matriz de correlación de Pearson:
                annual_inc  int_rate  loan_amnt  annual_inc_log
annual_inc        1.000000  0.050614   0.268672        0.702369
int_rate          0.050614  1.000000   0.308663        0.071698
loan_amnt         0.268672  0.308663   1.000000        0.438837
annual_inc_log    0.702369  0.071698   0.438837        1.000000


In [16]:
# Acà hacemos un analisis exploratorio de los datos con la libreria ydata_profiling de forma grafica
# pero debes asegurarte que la data este limpia, el tipo este correctamente definido, que lo que sea categorico sea categorico
# y lo numercio sea numerico, que no haya valores nulos, etc. de lo contrario el informe no se generara correctamente

# se debe ademas crear un environmet nuevo
# asegurarse de instalar:
#  pip install ipywidgets
#  pip install ydata-profiling
import pandas as pd
from ydata_profiling import ProfileReport

filename = "_data/Lending_club_cleaned_2.csv"
df = pd.read_csv(filename)

profile = ProfileReport(df,title='Informe de los datos', explorative = True)
profile.to_notebook_iframe()

Render HTML: 100%|██████████| 1/1 [00:00<00:00, 28.48it/s]
